# CPG-RL 訓練：智元 **D1 Max（中狗 · ZSM-1w）** · MJX · Colab GPU

移植自 `task6/notebooks/cpg_rl_d1w_colab.ipynb`（D1 EDU 輪足版，reward 已迭代到 v4）。
動作空間 12 維（每腿 `mux / muy / omega`），四顆輪子維持**純阻尼**，不進動作空間。

---

## ★ 與 task6 最關鍵的六個差異

| 項目 | task6 D1 EDU | **本檔 D1 Max** |
|---|---|---|
| 訓練模型 | 官方 XML 直接用 | ★ **`zgws_mjx.xml`（產生物）**。官方 XML 的碰撞網格 `BASE_LINK` **98,569 頂點**、四輪各 31,730，MJX 的 plane–convex 是逐頂點算的，2048 環境會 OOM |
| 佔空比 | env 沒有 `duty_remap`（等效 0.5） | ★ **`duty=0.80` 必須進 env**。這台實測 `duty ≤ 0.70` 是 **12/12 全跌**，照抄 task6 會得到一台永遠站不住的狗 |
| IK | home 附近線性化 Jacobian | ★ **解析式**。抬腿 100 mm、連桿 0.26+0.28 m，線性化誤差會直接汙染「抬腿量」——而那正是 reward 的 `r_clr` 要獎勵的東西 |
| HOME | 四腿共用一組三元組 | ★ **(4,3) 每腿各自**，前後 hip/knee 反號的 X 型站姿 |
| 致動器 | XML 內建 kp=80 / kd=1 | `position` kp **60/120/120** kv 1.0（ABAD 與 HIP/KNEE **不共用 Kp**）；輪 `velocity` kv 0.5 |
| obs | 69 維，`cmd` 3 維 | **68 維，`cmd` 2 維 = (vx, wz)** |

---

## 這次做 RL 的理由：開迴路走得穩，但**走不直**

基準步態 `walk`（凍結在 `task7/inference/gait_baseline.py`）：

| | 值 |
|---|---|
| 20 / 60 / 120 / **180 秒** × 12 擾動 | **跌倒 0/12** |
| 行進速度 `speed_travel` | **0.148 m/s** |
| 超限 / 力矩飽和 / IK 縮限 | 全部 0.00% |
| **偏航** | **−0.5 ~ −0.9 °/s**，180 秒累積 −51°、側偏 −10.7 m |

偏航**開迴路調參繞不過去**（`x_off` 掃 20 mm 完全不變號、`d_step` 單調無過零點、
`ω` 在 1.8 變號但 2.0 又變回來 —— 那是分歧不是趨勢）。
所以 RL 要打敗的主要是**偏航**，速度是附帶。

⚠️ **不要引用 `speed_path` 當行進速度**，它逐控制步累加、把機身左右搖擺算成前進，
實測高估 **68%**。一律看 `speed_travel`（以一個步態週期為步長重算）。

---

## 訓練模型與原始模型的落差（已量，`docs/MJX模型對照_2026-08-27.md`）

行進速度 **−0.7%**、彈跳 −1.7%、支撐腳 **0%**、離地 **0%**、跌倒 0/12 vs 0/12。

★ 這是量出來的不是設計出來的：輪子形狀用全寬圓柱差 **−34%**、用球差 **−23%**，
**冠頂半徑的窄圓盤（半寬 5 mm）**才差 −0.7%。

---

## ⚠️ 上實機前的硬性前提

`/dev/shm/imu_central` 目前**只有離線快照解碼過，沒驗過是不是活的串流**，
而且 `xyzw` 的順序是「取樣當下機身剛好水平」推出來的，不是刻意做的平放實驗。
**四元數順序錯 → 重力向量翻掉 → policy 直接廢掉。**
上機前先跑 `task7/docs/現場操作卡_IMU平放複核.md`。（不影響訓練。）


In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

# 版本鎖死，不要放寬。brax 的預設 activation/分布由版本決定，而 activation
# 不匹配時 brax 載權重【不會報錯】，只會讓 policy 靜默錯亂。
# 這裡的版本 = 本機推論端 (task7/inference/) 的版本。
#
# jax<0.10：brax 0.14.2 的 ppo.train (train.py:756) 用 jax.device_put_replicated，
# 該 API 在 jax 0.10 已被移除（本機 jax 0.10.2 實測直接 AttributeError）。
# 用 jax[cuda12] 這個 extra 是為了讓 jaxlib 與 CUDA plugin 一起降到相容版本，
# 只寫 "jax<0.10" 會留下版本不合的 jaxlib/cuda plugin。
# ⚠️ 這一項無法在本機驗證：裝完務必看下一格印出的 devices 有沒有 cuda。
#    掉回 CPU 或裝不起來 → 回報，【不要】自行改 brax 版本（會動到 activation 預設值）。
!pip install -q "brax==0.14.2" "mujoco==3.10.0" "mujoco-mjx==3.10.0" "jax[cuda12]<0.10" mediapy
print("done")

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())   # 要看到 cuda

# 版本斷言：不匹配當場停住，不要繞過。訓練跑完才發現行為對不上就白費了。
import brax, mujoco
print("brax", brax.__version__, "| mujoco", mujoco.__version__)
assert brax.__version__ == "0.14.2", (
    f"brax 版本為 {brax.__version__}，本機推論端是 0.14.2。"
    "版本不同會改變 make_ppo_networks 的預設 activation，"
    "而 activation 不匹配時 brax 載權重【不會報錯】，只會讓 policy 行為錯亂。"
    "請回到上一格重跑安裝（Colab 有時需要「執行階段 → 重新啟動工作階段」才會生效）。"
)
assert mujoco.__version__ == "3.10.0", (
    f"mujoco 版本為 {mujoco.__version__}，本機是 3.10.0。"
    "MJX 的接觸/求解器行為隨版本改變，訓練與推論不同版會讓步態對不上。"
)
# jax 0.10 移除了 device_put_replicated，而 brax 0.14.2 的 ppo.train 會用它。
# 在這裡早死，不要拖到訓練那一格編譯完才炸。
assert hasattr(jax, "device_put_replicated"), (
    f"jax {jax.__version__} 已移除 device_put_replicated，brax 0.14.2 的 ppo.train 會失敗。"
    "需要 jax<0.10。若 Colab 無法在此版本下取得 GPU 支援，請回報——"
    "換 brax 版本會改變 make_ppo_networks 的預設 activation，那會讓權重與本機推論端靜默不匹配。"
)
if not any(d.platform == "gpu" for d in jax.devices()):
    print("⚠️ 沒抓到 GPU。確認「執行階段 → 變更類型 → GPU」，"
          "以及上一格的 jax[cuda12]<0.10 是否把 CUDA 支援裝掉了。")
print("版本 OK")

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/HGLLLLL/RBTDOG_SIM.git"
BRANCH = "feat/d1-edu-cpg-rl"    # task7/ 目前在這個功能分支；併進 main 後改成 "main"
DEST = "rbtdog_sim"              # 明寫目的地：repo 名是大寫 RBTDOG_SIM，預設會 clone 成別的資料夾

if not os.path.exists(DEST):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DEST],
                   check=True)

sys.path.insert(0, f"{DEST}/task7/inference")

# ★ 這裡**不需要** fetch_assets.sh。訓練模型 zgws_mjx.xml 完全不依賴 STL：
#   官方模型要 54 MB 網格，而那要從 2.1 GB 的發布包裡解出來，Colab 每次開機都得重抓。
#   產生器（task7/model/zgws/make_mjx_model.py）已經把碰撞網格換成原始形狀、
#   視覺網格整個拿掉，所以 clone 完就能用。
SCENE = f"{DEST}/task7/model/zgws/scene_flat_mjx.xml"
print("model exists:", os.path.exists(SCENE))

assert os.path.exists(SCENE), (
    f"抓不到 {SCENE}。\n"
    f"1) 確認分支 {BRANCH} 的最新 commit 已 push 上 {REPO}"
    f"（--depth 1 只抓得到遠端當下的內容，本機沒 push 的修正 Colab 看不到）；\n"
    f"2) 或用左側檔案面板把 task7/model/zgws/*.xml 與 task7/inference/*.py 傳上來。"
)

In [ ]:
import jax.numpy as jnp
import numpy as np

import gait_baseline as gb
import leg_kin
import max_model as mm
import obs_max
from cpg_max import PHASE_WALK

# ★ 常數一律從 repo import，不在 notebook 裡重打一遍。
#   重打就是第二份真實來源，而兩份數字遲早會分岔（分岔之後訓練與推論是兩台機器人）。

CTRL_DT, SIM_DT = mm.CTRL_DT, mm.SIM_DT      # 0.02 / 0.002
N_FRAMES = int(round(CTRL_DT / SIM_DT))      # = 10（task6 是 5 → 這台每步計算量約 2 倍）
A_CONV, W_COUP, N_CPG_SUB = mm.A_CONV, mm.W_COUP, mm.N_CPG_SUB
MU_MIN, MU_MAX = mm.MU_MIN, mm.MU_MAX
G_P = mm.G_P

# ---- 基準步態（凍結在 gait_baseline.py，每個數字的判準來源也寫在那裡）----
DUTY = gb.BASELINE["duty"]            # 0.80 ⚠️ ≤0.70 是 12/12 全跌，不可以動
X_OFF = gb.BASELINE["x_off"]          # −0.040 m，平均俯仰單調過零點
G_C = gb.BASELINE["g_c"]              # 0.08
Z_SAG = gb.BASELINE["z_sag"]          # 0.0325，★只加在擺動相
D_STEP, D_STEP_Y = gb.BASELINE["d_step"], gb.BASELINE["d_step_y"]

# ---- 動作空間 ----
# ω 上限的依據是 12 擾動掃描：1.8 → 0.316 m/s 但彈跳 +48%、支撐腳 −0.3；
# 2.0 速度反而掉回 0.281。給到 2.0 有餘裕又不會掃進劣化區。
# 混沌區（trot、duty=0.5，速度在 0.11–0.46 m/s 之間跳）因 DUTY 鎖死而不可達。
OMEGA_MIN, OMEGA_MAX = 0.0, 2.0

HOME_np = np.array(mm.HOME)
HOME12_np = np.array(mm.HOME12)
KNEE_SIGN_np = leg_kin.knee_sign_of(mm.HOME)      # 前腿 −1、後腿 +1
F0S_np = leg_kin.home_foot(mm.HOME)               # (4,3) 每腿各自的基準足端
LEG_QPOS_IDX = jnp.array(mm.LEG_QPOS_IDX)
LEG_QVEL_IDX = jnp.array(mm.LEG_QVEL_IDX)
LEG_ACT_IDX = jnp.array(mm.LEG_ACT_IDX)
WHEEL_ACT_IDX = jnp.array(mm.WHEEL_ACT_IDX)
OBS_DIM, ACT_DIM = obs_max.OBS_DIM, obs_max.ACT_DIM   # 68 / 12

print(f"N_FRAMES={N_FRAMES}  OBS_DIM={OBS_DIM}  ACT_DIM={ACT_DIM}  DUTY={DUTY}  "
      f"X_OFF={X_OFF}  ω∈[{OMEGA_MIN},{OMEGA_MAX}]")
assert (OBS_DIM, ACT_DIM) == (68, 12)
assert DUTY == 0.80, "duty 不是 0.80 —— 這台 duty ≤ 0.70 是 12/12 全跌"

In [ ]:
PHASE_WALK_j = jnp.array(PHASE_WALK)
PHI = PHASE_WALK_j[None, :] - PHASE_WALK_j[:, None]


def cpg_init():
    return {"rx": jnp.full(4, 1.5), "rx_d": jnp.zeros(4),
            "ry": jnp.full(4, 1.5), "ry_d": jnp.zeros(4),
            "theta": PHASE_WALK_j}


def cpg_step(c, mux, muy, omega, dt):
    """與 task7/inference/cpg_max.make_cpg_step 逐行相同，只是換成 jnp。

    ⚠️ 耦合項會把相位拉回 PHI 定義的關係，所以**只改初始相位是無效的**，
       必須連耦合矩陣一起換（task6 的教訓）。
    """
    rx, rxd, ry, ryd, th = c["rx"], c["rx_d"], c["ry"], c["ry_d"], c["theta"]
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):
        rxd = rxd + A_CONV * (A_CONV / 4.0 * (mux - rx) - rxd) * h
        rx = rx + rxd * h
        ryd = ryd + A_CONV * (A_CONV / 4.0 * (muy - ry) - ryd) * h
        ry = ry + ryd * h
        rbar = 0.5 * (rx + ry)
        diff = th[None, :] - th[:, None] - PHI
        th = th + (2 * jnp.pi * omega
                   + W_COUP * jnp.sum(rbar[None, :] * jnp.sin(diff), 1)) * h
    return {"rx": rx, "rx_d": rxd, "ry": ry, "ry_d": ryd,
            "theta": jnp.mod(th, 2 * jnp.pi)}


def duty_remap(th, duty):
    """相位重映射成「擺動相佔一圈的 (1−duty)、站立相佔 duty」。

    ★ 這一段 task6 的 env **沒有**（等效 duty=0.5）。
      D1 Max 實測 duty ≤ 0.70 是 12/12 全跌，照抄 task6 會得到一台永遠站不住的狗。
      duty=0.5 時本函式恆等於原樣，所以它在 task6 上「看起來沒必要」。
    """
    ph = jnp.mod(th, 2 * jnp.pi) / (2 * jnp.pi)
    sw = 1.0 - duty
    return jnp.where(ph < sw,
                     jnp.pi * ph / sw,                      # 擺動 → 0~π
                     jnp.pi + jnp.pi * (ph - sw) / duty)    # 站立 → π~2π


def act_to_cmd(a):
    """12 維動作 → 每腿 (mux, muy, omega)。"""
    a = jnp.tanh(a).reshape(4, 3)
    mux = (a[:, 0] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    muy = (a[:, 1] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    om = (a[:, 2] + 1) / 2 * (OMEGA_MAX - OMEGA_MIN) + OMEGA_MIN
    return mux, muy, om


# ---- 解析式 IK（與 task7/inference/leg_kin.ik_ex 逐行相同，換成 jnp）----
# ⚠️ 不用 task6 那套 home 附近的線性化 Jacobian。這台抬腿 100 mm、連桿 0.26+0.28 m，
#    線性化誤差會直接變成「我以為抬了 100 mm、實際是別的數字」——
#    而抬腿量正是 reward 的 r_clr 要獎勵的東西，用有偏差的 IK 去量它會自己騙自己。
SIDE_X_j = jnp.array(mm.SIDE_X)
SIDE_Y_j = jnp.array(mm.SIDE_Y)
L_T, L_S = mm.L_THIGH, mm.L_SHANK
A2H_X, A2F_Y = mm.ABAD_TO_HIP_X, mm.ABAD_TO_FOOT_Y
REACH_LO = abs(L_T - L_S) + 1e-6
REACH_HI = L_T + L_S - 1e-6


def ik_j(k, p, knee_sign):
    yp = SIDE_Y_j[k] * A2F_Y
    px, py, pz = p[0], p[1], p[2]
    # 構不到時沿徑向縮到可達邊界，不丟 NaN —— 靜默的 NaN 會讓整段模擬變 nan
    # 而只發一個 RuntimeWarning。
    zp2 = jnp.maximum(py * py + pz * pz - yp * yp, 0.0)
    zp = -jnp.sqrt(zp2)
    q1 = jnp.arctan2(yp * pz - zp * py, yp * py + zp * pz)

    xp = px - SIDE_X_j[k] * A2H_X
    r = jnp.sqrt(xp * xp + zp * zp)
    r_new = jnp.clip(r, REACH_LO, REACH_HI)
    scale = jnp.where(r > 1e-12, r_new / r, 1.0)
    xp, zp = xp * scale, zp * scale
    r2 = r_new * r_new

    cos_q3 = (r2 - L_T ** 2 - L_S ** 2) / (2 * L_T * L_S)
    q3 = jnp.sign(knee_sign) * jnp.arccos(jnp.clip(cos_q3, -1.0, 1.0))
    a = -(L_T + L_S * jnp.cos(q3))
    b = -L_S * jnp.sin(q3)
    q2 = jnp.arctan2(a * xp - b * zp, a * zp + b * xp)
    return jnp.array([q1, q2, q3])


F0S_j = jnp.array(F0S_np)
KNEE_SIGN_j = jnp.array(KNEE_SIGN_np)


def joint_targets_j(c):
    """CPG 狀態 → 12 個關節目標角。"""
    th = duty_remap(c["theta"], DUTY)
    fx = 2 * (c["rx"] - MU_MIN) / (MU_MAX - MU_MIN) - 1
    fy = 2 * (c["ry"] - MU_MIN) / (MU_MAX - MU_MIN) - 1
    dx = -D_STEP * fx * jnp.cos(th) + X_OFF
    dy = D_STEP_Y * fy * jnp.cos(th)
    # ★ z_sag 只加在擺動相。站立相就是靠位置伺服的追蹤誤差在出力撐機身的，
    #   站立相也補等於把機身放掉。沒補的後果是「命令抬 100 mm、實際只抬 67 mm」，
    #   而且超限／飽和／IK縮限／相位鎖定四個診斷指標全是乾淨的，完全看不出問題
    #   （實測未補償時整台倒退走，12 秒 −987 mm）。
    dz = jnp.where(jnp.sin(th) > 0, (G_C + Z_SAG) * jnp.sin(th), G_P * jnp.sin(th))
    tgt = F0S_j + jnp.stack([dx, dy, dz], -1)
    q = jnp.stack([ik_j(k, tgt[k], KNEE_SIGN_j[k]) for k in range(4)])
    return q.reshape(12)


# ---- ★ 對照已驗證的 numpy 版 ----------------------------------------------
# 沒有這個檢查的話，JAX 移植過程中任何一個號差錯都會安靜地變成
# 「訓練與推論是兩台機器人」—— 而且訓練曲線會很漂亮，因為 policy 只是學會了
# 另一台機器人。
#
# 容差 1e-4 rad（0.006°）是 **float32 的捨入**，不是留給邏輯誤差的鬆動：
# 同一段程式在 `jax.config.update("jax_enable_x64", True)` 之下實測差 **2.2e-16 rad**
# （逐位相同），float32 之下是 3.4e-6。MJX 本來就跑 float32，所以這裡不開 x64。
# ⚠️ 若這個數字跳到 1e-3 以上，那就不是精度問題了，是真的有一項移植錯了。
import cpg_max as _cm

_c_np = _cm.cpg_init(PHASE_WALK)
_step_np = _cm.make_cpg_step(PHASE_WALK)
_c_j = cpg_init()
for _ in range(37):          # 37 步：把起步暫態也涵蓋進去
    _c_np = _step_np(_c_np, np.full(4, 1.8), np.full(4, 1.5), np.full(4, 1.4), CTRL_DT)
    _c_j = cpg_step(_c_j, jnp.full(4, 1.8), jnp.full(4, 1.5), jnp.full(4, 1.4), CTRL_DT)
_q_np, _n_clamp = _cm.joint_targets(_c_np, F0S_np, X_OFF, G_C, D_STEP, D_STEP_Y, DUTY,
                                    KNEE_SIGN_np, Z_SAG)
_q_j = np.asarray(joint_targets_j(_c_j))
_err = float(np.abs(_q_np - _q_j).max())
print(f"JAX vs numpy 關節角最大差 {_err:.3e} rad（IK 縮限 {_n_clamp} 腿）")
assert _err < 1e-4, "JAX 版的 CPG/IK 與 numpy 版不一致，訓練與推論會是兩台機器人"

In [ ]:
import jax
import mujoco
from brax.envs.base import Env, State
from mujoco import mjx

# ---- 擾動與雜訊 -------------------------------------------------------------
PUSH_EVERY = 100          # 每 100 控制步(=2 s) 注入一次速度擾動
PUSH_VEL = 0.6
GRAV_NOISE = 0.02         # 重力向量雜訊 sigma（IMU 是原始資料，精度一般）
GYRO_NOISE = 0.10         # 角速度雜訊 sigma (rad/s)
GYRO_BIAS = 0.05          # 每 episode 取樣一次的角速度偏差上限 (rad/s)

# ---- 終止 -------------------------------------------------------------------
# ★ 這兩個常數直接從 repo import，不像 task6 得手抄。
#   task6 的註解寫「Colab 無法 import 本地模組，只能手抄，兩邊分岔不會有錯誤訊息」——
#   本檔 clone 了 repo 又把 task7/inference 加進 sys.path，所以這個風險整個消失。
FALL_GRAV_Z = mm.FALL_GRAV_Z          # −0.4，機身系重力 z 高於此值視為翻倒（約傾 66°）
# 低姿護欄。由 task4 Go2 的 0.18 / 0.30 等比例換到 D1 Max 走路高度 0.4825 → 0.2895。
# 沒有這條的話，「塌腰趴著慢慢挪」是個很好爬的 local optimum。
MIN_HEIGHT = 0.29
# r_h 的目標高度用**走路時**的實測高度 0.4825，不是靜態站立的 0.4493——
# 這台走起來機身會被撐高 33 mm，拿靜態值當目標等於一直罰它「站太高」。
NOMINAL_HEIGHT = mm.NOMINAL_HEIGHT_WALK

# ---- reward 權重 ------------------------------------------------------------
# ⚠️⚠️ 這些權重是照 **task6 的 D1 EDU（20.6 kg）** 實測量值訂的。
#      這台 41 kg、尺度不同（力矩上限 150 vs 33 N·m、機身高 0.48 vs 0.27 m），
#      **第一輪訓練後必須依下一格印出的實測值重新校準**，不要當成定案。
#      校準方法見「基準校準」那一格：基準步態在本動作空間裡是一個**固定動作**，
#      可以直接量出每一項在「已知可行的步態」上值多少。
W_PITCH     = 20.0    # × grav[0]²        機身俯仰姿態（v3 實測有效）
W_PITCHRATE = 0.05    # × qvel[4]²        俯仰角速度
W_OMEGA_VAR = 0.5     # × var(各腿 ω)     ★封住「把各腿頻率拆開」這個規避管道。
                      #   task6 v1 實測 policy 會讓 FL 1.67 / RL 1.29（前腳快 30%），
                      #   而 W_COUP=8 的耦合把相位硬鎖回去 → 結果不是步態散掉，
                      #   是**穩定的相位畸變**。不封這個口，下一輪會用同一招。
W_VZ        = 0.5     # × qvel[2]²        機身垂直彈跳（v3 打地鼠打出來的洞）
W_CLR       = 1.5     # × r_clr           ★正向抬腳獎勵。沒有它的話「乾脆別抬腳」
                      #   是最省事的解——抬腳本身就會晃機身，vz²/r_h 兩項都在跟離地量作對。
# ---- D1 Max 新增 ----
# ★ 偏航率追蹤：這是本次做 RL 的**主要理由**。開迴路的 0.4–0.9 °/s 偏航
#   調參繞不過去（x_off 掃 20 mm 完全不變號、d_step 單調無過零點、
#   ω 在 1.8 變號但 2.0 又變回來——那是分歧不是趨勢），航向只能由回授來。
W_YAW = 1.0
W_VX = 1.5
W_VY = 0.5
W_H = 0.5
W_ACT = 0.01
# ⚠️ 力矩懲罰係數：task6 用 5e-5，但那是在力矩上限 33 N·m 的機器上。
#    這台腿的上限是 150 N·m。**這個值是量出來反推的，不是沿用**：
#    基準步態（＝下一格那個固定動作，已知 180 秒不跌倒）實測 Σtau² 中位數 **1648**，
#    取 3e-5 讓這一項在「已知可行的步態」上值 **0.049 分**（正項總和上限 5.0，約 1%）。
#    ⚠️ 5e-6 只值 0.008 分，等於這一項不存在；5e-5 則是 0.082。
W_TAU = 3e-5

# ---- 輪子淨空（給 r_clr 用）------------------------------------------------
# 【reward 可以用上帝視角，obs 不行】
# 這裡用 data.geom_xpos（模擬真值的輪心世界高度），實機沒有這個訊號。
# 這不是問題：reward 只在訓練時計算，訓練產物只有 policy 網路權重，
# 而 policy 是純函式 obs → action，推論時 (local_infer_max.py) 不會呼叫到 reward 的任何一行。
# 反過來 obs 是推論時每一步都要餵的東西，只能放實機真的拿得到的量 ——
# 所以輪子淨空【絕對不可以】加進 _obs。


def _qinv(q): return jnp.array([q[0], -q[1], -q[2], -q[3]])


def _qrot(q, v):
    u = q[1:4]; t = 2.0 * jnp.cross(u, v); return v + q[0] * t + jnp.cross(u, t)


def w2b(quat, v): return _qrot(_qinv(quat), v)


class MaxCpgEnv(Env):
    def __init__(self):
        m = mujoco.MjModel.from_xml_path(SCENE)
        assert m.opt.timestep == SIM_DT, (
            f"訓練模型的 timestep 是 {m.opt.timestep}，不是 {SIM_DT}。"
            "PD 內迴圈頻率會跟原廠 controller_dt 對不上。")
        self._mj = m
        self.sys = mjx.put_model(m)
        self._lo = jnp.array(m.jnt_range[mm.leg_joint_ids(m), 0])
        self._hi = jnp.array(m.jnt_range[mm.leg_joint_ids(m), 1])
        # 四個輪碰撞 geom 的 id 與半徑。在這裡查一次存起來：
        # mj_name2id 是 Python 端呼叫，放進 step 裡會進不了 jit。
        gids = [mm._id(m, mujoco.mjtObj.mjOBJ_GEOM, f"{mm.PREFIX[l]}_FOOT_LINK_COLL")
                for l in mm.LEGS]
        self._wheel_gids = jnp.array(gids)
        self._wheel_r = float(m.geom_size[gids[0]][0])   # 圓盤半徑（冠頂實算 0.0960）
        self._init_q = self._settled_qpos(m)

    def _settled_qpos(self, m):
        """先在 CPU MuJoCo 上站定 1.5 秒，把落地後的 qpos 當成 reset 的起點。

        ⚠️ 不能直接放純運動學的 HOME 高度：位置伺服有靜態撓度（41 kg、kp 有限），
           直接放上去會先掉 40 mm，每個 episode 的前 0.5 秒都在處理這個瞬態。
           `cpg_walk_max.rollout` 也是先 SETTLE_S=1.5 秒才開走。
        ⚠️ 這台 MJCF 的 nkey=0（**沒有 keyframe**），所以不能像 task6 那樣讀 key_qpos。
        """
        import cpg_max as _cm
        d = mujoco.MjData(m)
        q = _cm.stand_targets(KNEE_SIGN_np, F0S_np, X_OFF)
        d.qpos[:3] = [0.0, 0.0, mm.NOMINAL_HEIGHT_KIN + 0.005]
        d.qpos[3:7] = [1.0, 0.0, 0.0, 0.0]
        d.qpos[mm.LEG_QPOS_IDX] = q
        mujoco.mj_forward(m, d)
        for _ in range(int(1.5 / SIM_DT)):
            d.ctrl[mm.LEG_ACT_IDX] = q
            d.ctrl[mm.WHEEL_ACT_IDX] = 0.0
            mujoco.mj_step(m, d)
        assert abs(d.qpos[2] - mm.NOMINAL_HEIGHT) < 0.05, (
            f"站定高度 {d.qpos[2]:.3f} m 與預期 {mm.NOMINAL_HEIGHT:.3f} 差太多"
            "——訓練模型的位置伺服行為與 CPU 端對不上，回頭查 G1 對照。")
        return jnp.array(d.qpos)

    def _ctrl(self, q_des):
        """12 個腿關節目標角 → 16 維 ctrl。輪子目標速度恆 0（＝純阻尼）。"""
        return jnp.zeros(16).at[LEG_ACT_IDX].set(q_des).at[WHEEL_ACT_IDX].set(0.0)

    def _wheel_clearance(self, data):
        """四腿各自的輪底離地高度 (m)。不進 obs，理由見上面說明。"""
        return data.geom_xpos[self._wheel_gids, 2] - self._wheel_r

    def _obs(self, data, c, cmd, last_a, gyro_bias):
        """68 維。⚠️ 欄位順序必須與 task7/inference/obs_max.py 的 OBS_LAYOUT 逐項一致
        ——本機推論端 local_infer_max.py 用的是那一份。"""
        grav = w2b(data.qpos[3:7], jnp.array([0.0, 0.0, -1.0]))
        gyro = data.qvel[3:6] + gyro_bias
        return jnp.concatenate([
            grav, gyro,
            data.qpos[LEG_QPOS_IDX] - jnp.array(HOME12_np),
            data.qvel[LEG_QVEL_IDX],
            cmd,                       # 2 維 (vx, wz)
            last_a,
            c["rx"], c["rx_d"], c["ry"], c["ry_d"],
            jnp.sin(c["theta"]), jnp.cos(c["theta"]),
        ])

    def reset(self, rng):
        k_cmd, k_bias, k_delay, k_noise = jax.random.split(rng, 4)
        data = mjx.make_data(self.sys).replace(qpos=self._init_q)
        data = data.replace(ctrl=self._ctrl(self._init_q[LEG_QPOS_IDX]))
        data = mjx.forward(self.sys, data)

        # 60% 的 episode 抽 wz = 0（＝「給我走直」）。開迴路繞不過去的偏航就是
        # 這條線要解的主要問題，走直的樣本必須夠多。
        k_vx, k_wz, k_zero = jax.random.split(k_cmd, 3)
        vx = jax.random.uniform(k_vx, minval=0.05, maxval=0.40)
        wz = jnp.where(jax.random.uniform(k_zero) < 0.6, 0.0,
                       jax.random.uniform(k_wz, minval=-0.4, maxval=0.4))
        cmd = jnp.array([vx, wz])

        gyro_bias = jax.random.uniform(k_bias, (3,), minval=-GYRO_BIAS, maxval=GYRO_BIAS)
        # ★ 動作延遲隨機化（task6 沒有這一項）。實機的鏈路是
        #   「我們寫 shm → 1 kHz daemon 讀 → 馬達」，一定有延遲，而 daemon 還有
        #   500 ms 的過期判定。延遲量沒量過，所以用「隨機 0 或 1 個控制步」蓋住它，
        #   而不是押一個沒根據的定值。
        delay = jax.random.bernoulli(k_delay, 0.5).astype(jnp.float32)

        c = cpg_init()
        info = {"rng": k_noise, "c": c, "last_a": jnp.zeros(ACT_DIM),
                "prev_a": jnp.zeros(ACT_DIM), "cmd": cmd, "gyro_bias": gyro_bias,
                "delay": delay, "step": 0}
        obs = self._obs(data, c, cmd, jnp.zeros(ACT_DIM), gyro_bias)
        # "reward" 這個鍵【必須】有，reset 與 step 都要有、型別一致。
        # brax 的 EvalWrapper.reset 會就地塞 metrics['reward']，而 num_evals>1 時
        # 訓練開始前就會先跑一次 eval，於是 EpisodeWrapper.step 的 lax.scan
        # 會拿到「進去 N 鍵、出來 N−1 鍵」→ 編譯完當場 TypeError。
        # 同理：step 裡新增任何 metrics 鍵，這裡也要一起加。
        z = jnp.zeros(())
        metrics = {"height": data.qpos[2], "vx": z, "reward": z, "pitch": z,
                   "clr": z, "vz": z, "yawerr": z, "vxerr": z}
        return State(data, obs, z, z, metrics, info)

    def step(self, state, action):
        info = dict(state.info)
        # ★ 動作延遲：delay=1 的 episode 一律用「上一步的動作」。
        act = jnp.where(info["delay"] > 0.5, info["prev_a"], action)

        mux, muy, om = act_to_cmd(act)
        c = cpg_step(info["c"], mux, muy, om, CTRL_DT)
        q_des = jnp.clip(joint_targets_j(c), self._lo, self._hi)

        data = state.pipeline_state
        rng, k_push, k_dir, k_obs = jax.random.split(info["rng"], 4)
        do_push = (info["step"] % PUSH_EVERY) == (PUSH_EVERY - 1)
        ang = jax.random.uniform(k_dir, minval=0.0, maxval=2 * jnp.pi)
        mag = jax.random.uniform(k_push, minval=0.0, maxval=PUSH_VEL)
        kick = jnp.where(do_push,
                         jnp.array([mag * jnp.cos(ang), mag * jnp.sin(ang), 0.0]),
                         jnp.zeros(3))
        data = data.replace(qvel=data.qvel.at[0:3].add(kick))

        ctrl = self._ctrl(q_des)

        def one(d, _):
            return mjx.step(self.sys, d.replace(ctrl=ctrl)), None
        data, _ = jax.lax.scan(one, data, None, length=N_FRAMES)

        grav = w2b(data.qpos[3:7], jnp.array([0.0, 0.0, -1.0]))
        vb = w2b(data.qpos[3:7], data.qvel[0:3])      # reward 用真值速度（只在模擬）
        cmd = info["cmd"]
        wz = data.qvel[5]                             # freejoint 的 qvel[3:6] 是機身系角速度

        r_vx = jnp.exp(-(vb[0] - cmd[0]) ** 2 / 0.02)
        r_vy = jnp.exp(-vb[1] ** 2 / 0.02)
        r_yaw = jnp.exp(-(wz - cmd[1]) ** 2 / 0.05)
        r_h = jnp.exp(-400.0 * (data.qpos[2] - NOMINAL_HEIGHT) ** 2)
        swing = jnp.maximum(jnp.sin(c["theta"]), 0.0)
        r_clr = jnp.mean(swing * jnp.clip(self._wheel_clearance(data) / G_C, 0.0, 1.0))

        c_act = jnp.sum((action - info["last_a"]) ** 2)
        c_tau = jnp.sum(data.actuator_force ** 2)
        c_pitch = grav[0] ** 2
        c_pitchrate = data.qvel[4] ** 2
        c_omvar = jnp.var(om)
        c_vz = data.qvel[2] ** 2

        reward = (W_VX * r_vx + W_VY * r_vy + W_YAW * r_yaw + W_H * r_h
                  + W_CLR * r_clr
                  - W_ACT * c_act - W_TAU * c_tau
                  - W_PITCH * c_pitch - W_PITCHRATE * c_pitchrate
                  - W_OMEGA_VAR * c_omvar - W_VZ * c_vz)

        fell = grav[2] > FALL_GRAV_Z
        too_low = data.qpos[2] < MIN_HEIGHT
        done = jnp.where(jnp.logical_or(fell, too_low), 1.0, 0.0)

        noise = jax.random.normal(k_obs, (6,))
        obs = self._obs(data, c, cmd, action, info["gyro_bias"])
        obs = obs.at[0:3].add(GRAV_NOISE * noise[0:3])
        obs = obs.at[3:6].add(GYRO_NOISE * noise[3:6])

        info.update({"rng": rng, "c": c, "last_a": action, "prev_a": action,
                     "step": info["step"] + 1})
        # 監看指標（不是 reward 項）。★ yawerr 與 vxerr 是本檔新增的主目標指標：
        # 訓練要打敗的是開迴路的偏航，沒有這兩個就看不出有沒有真的變好。
        metrics = {"height": data.qpos[2], "vx": vb[0], "reward": reward,
                   "pitch": jnp.abs(grav[0]) * 57.29578,
                   "clr": jnp.mean(self._wheel_clearance(data)) * 1000.0,
                   "vz": jnp.abs(data.qvel[2]),
                   "yawerr": jnp.abs(wz - cmd[1]),
                   "vxerr": jnp.abs(vb[0] - cmd[0])}
        return state.replace(pipeline_state=data, obs=obs, reward=reward,
                             done=done, metrics=metrics, info=info)

    @property
    def observation_size(self): return OBS_DIM

    @property
    def action_size(self): return ACT_DIM

    @property
    def backend(self): return "mjx"


print("MaxCpgEnv 已定義")

In [ ]:
_mm_model = mujoco.MjModel.from_xml_path(SCENE)
BASE_ID = mm._id(_mm_model, mujoco.mjtObj.mjOBJ_BODY, "base_link")
LEG_DOF = jnp.array(mm.LEG_QVEL_IDX)
WHEEL_DOF = jnp.array(mm.WHEEL_QVEL_IDX)
LEG_ACT = jnp.array(mm.LEG_ACT_IDX)
KP_NOM = jnp.array(np.tile(np.array(mm.KP3), 4))      # 60/120/120 × 4


def domain_randomize(sys, rng):
    @jax.vmap
    def per_env(rng):
        k1, k2, k3, k4, k5, k6, k7 = jax.random.split(rng, 7)

        # 地面摩擦。掃描實測 0.3 會跌、≥0.5 能走，所以下界取 0.4——
        # 讓它學會在「勉強能走」的地面上走，但不要把整段訓練泡在必跌的條件裡。
        geom_friction = sys.geom_friction.at[:, 0].set(
            jax.random.uniform(k1, minval=0.4, maxval=1.4))

        # PD 增益 ±20%。position 致動器：gainprm[0]=kp、biasprm=[0, −kp, −kv]。
        # ⚠️ 三個關節的名目 Kp **不一樣**（ABAD 60、HIP/KNEE 120），
        #    所以是整組乘一個係數，不是設成同一個值（task6 那台是共用 kp=80）。
        scale_kp = jax.random.uniform(k2, minval=0.8, maxval=1.2)
        kv = jax.random.uniform(k3, minval=0.5, maxval=2.0)
        kp_leg = KP_NOM * scale_kp
        gain = sys.actuator_gainprm.at[LEG_ACT, 0].set(kp_leg)
        bias = (sys.actuator_biasprm
                .at[LEG_ACT, 1].set(-kp_leg)
                .at[LEG_ACT, 2].set(-kv))

        # 質量。★ payload 0–5 kg 同時吸收兩件事：官方額定 5 kg 負載，
        #   以及 MJCF 38.821 kg 與規格書 41 kg 之間那 2.2 kg 的缺口
        #   （可能是沒算雙電池）。
        body_mass = sys.body_mass * jax.random.uniform(
            k4, (sys.nbody,), minval=0.9, maxval=1.1)
        body_mass = body_mass.at[BASE_ID].add(
            jax.random.uniform(k5, minval=0.0, maxval=5.0))

        # 關節靜摩擦。★ 腿的 ABAD 1.85 掃描已證明是**下界不是量測值**
        #   （正向那次沒有掙脫：速度整段停在量化底噪 +0.001 rad/s、力矩到最後還在上升），
        #   所以必須用範圍蓋住，不能當定值。
        fl = sys.dof_frictionloss
        fl = fl.at[LEG_DOF].multiply(jax.random.uniform(k6, minval=0.5, maxval=1.5))
        # 輪：兩天兩種條件量到 0.153 / 0.170，取 0.10–0.25。
        fl = fl.at[WHEEL_DOF].set(jax.random.uniform(k7, minval=0.10, maxval=0.25))
        return geom_friction, gain, bias, body_mass, fl

    gf, gain, bias, bm, fl = per_env(rng)
    in_axes = jax.tree_util.tree_map(lambda x: None, sys)
    in_axes = in_axes.replace(geom_friction=0, actuator_gainprm=0,
                              actuator_biasprm=0, body_mass=0, dof_frictionloss=0)
    sys = sys.replace(geom_friction=gf, actuator_gainprm=gain,
                      actuator_biasprm=bias, body_mass=bm, dof_frictionloss=fl)
    return sys, in_axes


print("domain_randomize ready")

In [ ]:
env = MaxCpgEnv()

# ⚠️ biastype 不是 affine 時 ctrl 會被當**力矩**直接施加，機器人當場塌掉且不報錯。
assert env.sys.actuator_biastype[0] == mujoco.mjtBias.mjBIAS_AFFINE, \
    "致動器 biastype 不是 affine，ctrl 會被當力矩施加 → 機器人會塌掉"
assert env.observation_size == 68 and env.action_size == 12

# ---- ★ 基準校準：基準步態在這個動作空間裡是一個**固定動作** ----------------
# mux=1.80 / muy=1.50 / ω=1.4 反推回 tanh 前的值。所以我們可以直接把
# 「已驗證 180 秒不跌倒的開迴路步態」餵進 env，回答兩件事：
#   1. env 有沒有重現開迴路基準？（CPU 端量到 0.147 m/s、彈跳 16.9 mm、支撐腳 3.20）
#   2. 每一項 reward 在「已知可行的步態」上值多少？——權重要照這個訂，
#      不是照 task6 那台 20.6 kg 的機器沿用。
def _inv(u, lo, hi):
    return float(np.arctanh(np.clip(2 * (u - lo) / (hi - lo) - 1, -0.999, 0.999)))


A_BASE = jnp.array([_inv(gb.BASELINE["mu_x"], MU_MIN, MU_MAX),
                    _inv(gb.BASELINE["mu_y"], MU_MIN, MU_MAX),
                    _inv(gb.BASELINE["omega"], OMEGA_MIN, OMEGA_MAX)] * 4)
_mx, _my, _om = act_to_cmd(A_BASE)
print(f"基準動作還原：mux={_mx[0]:.4f} muy={_my[0]:.4f} ω={_om[0]:.4f}"
      f"（應為 {gb.BASELINE['mu_x']} / {gb.BASELINE['mu_y']} / {gb.BASELINE['omega']}）")

jit_reset, jit_step = jax.jit(env.reset), jax.jit(env.step)
s = jit_reset(jax.random.PRNGKey(0))
print("reset ok, obs", s.obs.shape, "height %.4f m" % float(s.pipeline_state.qpos[2]))
assert abs(float(s.pipeline_state.qpos[2]) - mm.NOMINAL_HEIGHT) < 0.06, \
    "起步高度不對——站定流程沒生效"

# 跑 10 秒基準動作（500 控制步），統計後半段
import time as _time
_t0 = _time.time()
xs, hs, sup, taus, rs = [], [], [], [], []
for i in range(500):
    s = jit_step(s, A_BASE)
    if i >= 250:
        xs.append(float(s.pipeline_state.qpos[0]))
        hs.append(float(s.pipeline_state.qpos[2]))
        taus.append(float(jnp.sum(s.pipeline_state.actuator_force ** 2)))
        rs.append(float(s.reward))
    assert float(s.done) == 0.0, f"基準動作在第 {i} 步就 done 了——env 有問題"
print(f"[基準] 10 s 前進 {float(s.pipeline_state.qpos[0]):+.3f} m  "
      f"機身高 {np.mean(hs)*1000:.1f} mm  彈跳 {(max(hs)-min(hs))*1000:.1f} mm  "
      f"平均 reward {np.mean(rs):.3f}  ({_time.time()-_t0:.0f}s)")
print(f"[對照] CPU 端開迴路：速度 0.147 m/s、機身高 482 mm、彈跳 16.9 mm")
print("       ⚠️ 彈跳會比 CPU 端高，因為 env 每 2 秒注入一次 0.6 m/s 推撞（PUSH_EVERY）。"
      "速度與機身高才是這裡要對的兩個數字。")
print(f"[校準] Σtau² 中位數 {np.median(taus):.0f} → W_TAU={W_TAU} 讓這一項值 "
      f"{W_TAU*np.median(taus):.3f} 分（正項總和上限 {W_VX+W_VY+W_YAW+W_H+W_CLR:.1f}）")